# 06 · Modelo final de v30 y comparación con lo enviado

`scripts/entrenar.py` con la receta de v30:

- Clasificador de copia: CatBoost Logloss, profundidad 8, lr 0,06 y 3.300 árboles, con todas las salidas aptas
  de 2025.
- Regresor del rodaje: CatBoost RMSE sobre `etiqueta − referencia NM/P10`, profundidad 10, lr 0,03 y 24.000 árboles.
- Fórmula `p·(despegue − programada) + (1 − p)·rodaje`, cola de Roma recalibrada (tramo de retraso × grupo de
  emparejamiento), consenso, proyección LOBT ± 3.606 s, suelo de 60 s, redondeo y orden de la plantilla.

Lo hacemos de dos maneras:

1. Con los modelos guardados de v30 (`Submissions/v30/clf.cbm` y `model.cbm`): rehace la matriz, la cola y la
   proyección, y predice.
2. Reentrenando desde cero (`REENTRENAR = 1`): mismo código y mismos datos, pero el modelo sale equivalente, no
   idéntico, porque CatBoost en GPU no es determinista (suma en paralelo y el orden cambia entre ejecuciones; dos
   entrenamientos con la misma semilla difieren ~11 s de media en las predicciones).

En modo final se predicen las 670.790 salidas de `final_submitting.parquet` (en `v30_final_modelos` y, si se
reentrena, en `v30_final`), después de comprobar la matriz de los cuatro meses. La comparación con v30 se hace sobre
las salidas de enero y julio que comparten ID, y no pueden salir iguales: sus horas cambian hasta ±12 s y las
agregaciones se rehacen con cuatro meses.

In [ ]:
import sys
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "scripts" / "cuaderno.py").exists())
sys.path[:0] = [str(ROOT / "scripts"), str(ROOT)]
from cuaderno import elegir_periodo, existen, interruptor, run  # noqa: E402

PERIODO = elegir_periodo("ranking")   # abierto suelto: "ranking" (ene + jul 2026) o "final" (ene, feb, jun y jul 2026)
FINAL = PERIODO == "final"
import periodo as PER  # noqa: E402  rutas del periodo: PER.RANKING (fichero a predecir), PER.FEAT (variables)
import os

NOMBRE = os.environ.get("V30_NOMBRE", "v30_final" if FINAL else "v30_recreado")
REENTRENAR = interruptor("REENTRENAR", True)
assert NOMBRE != "v30", "no sobrescribir la carpeta oficial de v30"
RECETA = ["fuentes,adsb,adsb_stand", "24000", "1", "3300", '{"learning_rate": 0.03}']
print(f"PERIODO = {PERIODO} · NOMBRE = {NOMBRE} · REENTRENAR = {REENTRENAR}")

## 0. Comprobación de la matriz (solo en modo final)

In [ ]:
if FINAL:   # mismas 177 columnas que 2025, filas = plantilla, cobertura por mes
    run("scripts/preparar_final.py", "comprobar")
else:
    print("PERIODO ranking: la matriz es la de v30")

## 1. Predicción con los modelos guardados de v30

In [ ]:
run("scripts/entrenar.py", f"{NOMBRE}_modelos", *RECETA, "--modelos=Submissions/v30")

## 2. Reentrenamiento desde cero

In [ ]:
if REENTRENAR:
    run("scripts/entrenar.py", NOMBRE, *RECETA)
else:
    print("Reentrenamiento omitido (REENTRENAR = 0)")

## 3. Comparación con el fichero enviado

In [ ]:
import hashlib
import json

import numpy as np
import pandas as pd

oficial_f = ROOT / "Submissions" / "v30" / "knowledgeable-alligator_v30.parquet"
sha = lambda p: hashlib.sha256(p.read_bytes()).hexdigest()
o = pd.read_parquet(oficial_f)


def comparar(nombre):
    """Fila a fila con el fichero enviado; con PERIODO final, solo sobre las salidas que comparten ID con él."""
    f = ROOT / "Submissions" / nombre / f"knowledgeable-alligator_{nombre}.parquet"
    n = pd.read_parquet(f)
    if not FINAL:
        assert o.MVT_ID_mvt.equals(n.MVT_ID_mvt), "orden de filas distinto"
    m = n.merge(o, on="MVT_ID_mvt", suffixes=("", "_v30"), validate="one_to_one")
    dif = m.TAXITIME_SEC_mvt.to_numpy(float) - m.TAXITIME_SEC_mvt_v30.to_numpy(float)
    return {"sha256": sha(f), "identico_al_enviado": sha(f) == sha(oficial_f),
            "filas": len(n), "filas_comunes_con_v30": len(m),
            "predicciones_iguales_%": round(100 * float(np.mean(dif == 0)), 2),
            "dif_media_s": round(float(dif.mean()), 2),
            "dif_abs_mediana_s": float(np.median(np.abs(dif))), "dif_abs_p99_s": float(np.percentile(np.abs(dif), 99)),
            "rmse_entre_ambos_s": round(float(np.sqrt(np.mean(dif ** 2))), 2),
            "media_s": round(float(n.TAXITIME_SEC_mvt.mean()), 2),
            "valido": json.loads((f.parent / "informe.json").read_text(encoding="utf-8"))["validacion"]["valido"]}


comparacion = {"periodo": PERIODO, "enviado": {"sha256": sha(oficial_f), "media_s": round(float(o.TAXITIME_SEC_mvt.mean()), 2)},
               "con_modelos_guardados": comparar(f"{NOMBRE}_modelos")}
if not FINAL:
    assert comparacion["con_modelos_guardados"]["identico_al_enviado"], "los modelos guardados no reproducen v30"
if REENTRENAR:
    comparacion["reentrenado"] = comparar(NOMBRE)
(ROOT / "Submissions" / f"{NOMBRE}_modelos" / "comparacion_v30.json").write_text(
    json.dumps(comparacion, indent=1, ensure_ascii=False), encoding="utf-8")
comparacion